# 🎮 Machine Learning Pipeline: Dự đoán Doanh số Trò chơi Điện tử (Video Game Sales)
**Nhóm thực hiện:** WEEBFORCE (Module 4 - AI & Data Processing)  
**Kiến trúc:** End-to-End Production ML Pipeline (Scikit-Learn)  
**Bộ dữ liệu:** `vgsales_clean.csv` (16,000+ tựa game lịch sử)

---

## 📌 1. Đặt vấn đề Kỹ thuật (Senior ML Formulation)
Trong ngành công nghiệp game, quyết định đầu tư một dự án phụ thuộc lớn vào việc ước tính doanh số trước khi phát hành (Pre-release Sales Forecasting).
- **Mục tiêu:** Dự đoán doanh số toàn cầu `Global_Sales` (triệu bản) dựa trên các đặc trưng tiền phát hành: `Platform`, `Genre`, `Publisher`, `Year`, v.v.
- **Các thách thức kỹ thuật:**
  1. **Extreme Right Skewness:** Doanh số game phân phối lệch phải rất nặng (phần lớn bán dưới 0.5M, vài game đạt > 30M). Áp dụng **Log Transformation** ($y_{log} = \ln(1 + y)$) để ổn định phương sai và tối ưu RMSLE.
  2. **High-Cardinality Categorical Data:** Hơn 570 nhà phát hành. Ta gom nhóm các hãng nhỏ ít xuất hiện thành `Other` để chống bùng nổ số chiều và Overfitting.
  3. **Data Leakage Prevention:** Đóng gói tiền xử lý và mô hình vào `sklearn.pipeline.Pipeline` và `ColumnTransformer`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("✅ Môi trường thư viện Machine Learning đã sẵn sàng!")

## 📊 2. Tải Dữ liệu & Kỹ thuật Tạo Đặc trưng (Feature Engineering)

In [ ]:
data_file = 'vgsales_clean.csv' if os.path.exists('vgsales_clean.csv') else 'vgsales.csv'
df = pd.read_csv(data_file)
df = df.dropna(subset=['Year', 'Platform', 'Genre', 'Publisher', 'Global_Sales']).copy()
df['Year'] = df['Year'].astype(int)
df = df[df['Year'] <= 2020].copy()

# Feature Engineering:
# 1. Độ dài tên game
df['Name_Length'] = df['Name'].apply(lambda x: len(str(x)))

# 2. Nhận diện game là phần tiếp theo (Sequel indicator: 2, 3, IV, ...)
sequel_pattern = r'(?:\b2\b|\b3\b|\b4\b|\b5\b|\b6\b|\b7\b|\b8\b|\b9\b|\bII\b|\bIII\b|\bIV\b|\bV\b|\bVI\b|\bVII\b|\bVIII\b|\bIX\b|\bX\b)'
df['Is_Sequel'] = df['Name'].str.contains(sequel_pattern, regex=True, case=False).astype(int)

# 3. Gom nhóm Publisher có ít hơn 15 game thành 'Other'
top_publishers = df['Publisher'].value_counts()[lambda x: x >= 15].index
df['Publisher_Cleaned'] = df['Publisher'].apply(lambda p: p if p in top_publishers else 'Other')

print(f"Tổng số mẫu: {len(df):,}")
print(f"Platforms: {df['Platform'].nunique()} | Genres: {df['Genre'].nunique()} | Publishers gộp: {df['Publisher_Cleaned'].nunique()}")
df[['Name', 'Platform', 'Genre', 'Publisher_Cleaned', 'Year', 'Is_Sequel', 'Global_Sales']].head(5)

## 🔬 3. Trực quan hóa Phân phối Target & Log-Transformation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Phân phối gốc
sns.histplot(df['Global_Sales'], bins=50, kde=True, ax=axes[0], color='#e74c3c')
axes[0].set_title('Phân phối Gốc (Extreme Right-Skew)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Doanh số toàn cầu (triệu bản)')

# Phân phối sau log1p
sns.histplot(np.log1p(df['Global_Sales']), bins=50, kde=True, ax=axes[1], color='#2ecc71')
axes[1].set_title('Phân phối sau Log1p (Gần chuẩn Gaussian)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('log(1 + Global_Sales)')

plt.tight_layout()
plt.show()

## 🛠️ 4. Xây dựng Pipeline Xử lý & Train/Test Split (80/20)

In [ ]:
cat_features = ['Platform', 'Genre', 'Publisher_Cleaned']
num_features = ['Year', 'Name_Length', 'Is_Sequel']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
        ('num', StandardScaler(), num_features),
    ],
    remainder='drop',
)

X = df[cat_features + num_features]
y = df['Global_Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print(f"Train size: {len(X_train):,} | Test size: {len(X_test):,}")

## 🚀 5. Huấn luyện & Đánh giá So sánh Đa mô hình (Model Leaderboard)

In [ ]:
candidate_models = {
    'Baseline (Dummy Mean)': DummyRegressor(strategy='mean'),
    'Ridge Regression (L2)': Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, max_depth=15, random_state=RANDOM_STATE, n_jobs=-1),
    'HistGradientBoosting Regressor': HistGradientBoostingRegressor(max_iter=150, max_depth=8, random_state=RANDOM_STATE),
}

benchmark_results = []
trained_pipelines = {}

for name, model in candidate_models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    pipeline.fit(X_train, y_train_log)
    trained_pipelines[name] = pipeline
    
    pred_log = pipeline.predict(X_test)
    pred_sales = np.clip(np.expm1(pred_log), a_min=0, a_max=None)
    
    r2_log = r2_score(y_test_log, pred_log)
    rmsle = np.sqrt(mean_squared_error(y_test_log, pred_log))
    mae = mean_absolute_error(y_test, pred_sales)
    rmse = np.sqrt(mean_squared_error(y_test, pred_sales))
    
    benchmark_results.append({
        'Model': name,
        'R2 (Log Scale)': round(r2_log, 4),
        'RMSLE': round(rmsle, 4),
        'MAE (Triệu bản)': round(mae, 4),
        'RMSE (Triệu bản)': round(rmse, 4)
    })

leaderboard = pd.DataFrame(benchmark_results).sort_values('RMSLE')
display(leaderboard)

## 🔍 6. Phân tích Tầm quan trọng Đặc trưng (Feature Importance)

In [ ]:
rf_model = trained_pipelines['Random Forest Regressor'].named_steps['regressor']
ohe = preprocessor.named_transformers_['cat']
cat_names = list(ohe.get_feature_names_out(cat_features))
all_features = cat_names + num_features

feat_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_df.head(15), x='Importance', y='Feature', hue='Feature', palette='viridis', legend=False)
plt.title('Top 15 Đặc trưng Ảnh hưởng Lớn nhất đến Doanh số Game', fontsize=14, fontweight='bold')
plt.xlabel('Điểm Tầm quan trọng (Importance Score)')
plt.tight_layout()
plt.show()

## 📦 7. Đóng gói Mô hình & Thử nghiệm Dự đoán Thực tế (Inference)

In [ ]:
best_pipeline = trained_pipelines['HistGradientBoosting Regressor']
joblib.dump(best_pipeline, 'best_game_sales_model.joblib')
print('✅ Đã xuất mô hình: best_game_sales_model.joblib')

def predict_game_sales(name, platform, genre, publisher, year):
    df_single = pd.DataFrame([{
        'Platform': platform,
        'Genre': genre,
        'Publisher_Cleaned': publisher if publisher in top_publishers else 'Other',
        'Year': year,
        'Name_Length': len(name),
        'Is_Sequel': int(bool(pd.Series([name]).str.contains(sequel_pattern, regex=True).iloc[0]))
    }])
    pred_log = best_pipeline.predict(df_single)
    sales = float(np.expm1(pred_log)[0])
    return max(0.0, sales)

# Kiểm thử thực tế các kịch bản
samples = [
    ('The Legend of Zelda: Next Gen', 'Wii', 'Action', 'Nintendo', 2016),
    ('Call of Duty: Black Ops Next', 'PS4', 'Shooter', 'Activision', 2016),
    ('Indie Pixel Roguelike', 'PC', 'Adventure', 'Indie Dev', 2015),
    ('FIFA 18 World Cup Edition', 'PS4', 'Sports', 'Electronic Arts', 2017)
]

for name, plat, gen, pub, yr in samples:
    res = predict_game_sales(name, plat, gen, pub, yr)
    print(f'🎮 {name:<30} | {plat:<4} | {gen:<10} | {pub:<18} ➔ Dự đoán: {res:.2f} triệu bản')